In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs
import preprocessing as pp
import candidates as ca

In [2]:
# UNCOMMENT If you run for the first time without the CSV ready -> gets firstbatch_2

# pp.transform_jsons()

In [3]:
df = pd.read_csv('algebrakit_firstbatch_2.csv')
collection_attributes = ['skills', 'stepTypes', 'event_types', 'event_evaluations', 'positive_skills', 'negative_skills']

for colatt in collection_attributes:
    df[colatt] = df[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df.head(2)

,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,...,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q1,...,1.0,1.0,1.0,True,2.0,0.0,False,0.0,0,0
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q2,...,0.0,0.0,1.0,False,2.0,2.0,True,0.0,1,2


# Data distribution in the Use Case
Our data can be split into two categories: the **exercise attributes** that describe the contents of an exercise and the **student attributes** that describe the workflow and performance of the students that made the exercise. The hierarchical level that each attribute is at, is denoted behind the attribute name.

**Exercise Attributes:**
* skills (eid)
* stepTypes (eid)
* marksTotal (eid)
* itemType (qid)
* interactionType (iid)
* solution (iid)
* content (iid)
* interaction marksTotal (iid)
* maxAttempts (iid)

**Student Attributes:**
* success (sid)
* finished (sid)
* marksEarned (sid)
* eventTypes (iid)
* eventEvaluations (iid)
* positive skills (iid)
* negative skills (iid)
* result status (iid)
* result progress (iid)
* interaction marksEarned (iid)
* interaction finished (iid)
* usedAttempts (iid)
* failed (iid)
* marksPenalty (iid)
* hintsRequested (iid)
* mathErrors (iid)

## Attribute Types and Descriptive and Target Space.
That we have this many attributes, does not mean that we select all attributes. We therefore select only a part of it. These attributes can be composed of six categories: numerical (float), ordinal (int), categorical, boolean, collection, sequence, and subsequence. The last three are list-type attributes that simulate the aggregation of a hierarchical level that has no need to be expanded into seperate rows in the data. 

**Target Attribute**
* finished (boolean)

The below mentioned are our descriptors:

**Numerical (float)**
* marksPenalty

**Ordinal (int)**
* marksTotal
* interaction marksTotal
* usedAttempts
* marksPenalty
* hintsRequested
* mathErrors

**Categorical**
* itemType
* interactionType

**Boolean**
* nvt

**Collection (list, no order)**
* skills
* positive skills
* negative skills

**Sequence (list, order immediate)**
* eventTypes
* eventEvaluations

**Subsequence (list, order with pauses)**
* stepTypes (also possible as collection or sequence)

In [4]:
cats = rf.subsequence_refinement(df, "event_types", 3)

In [5]:
# we split of the data into two spaces
descriptors = ['itemType', 'interactionType','marksPenalty','marksTotal', 'marksTotal_interaction', 'usedAttempts', 'marksPenalty', 'hintsRequested', 'mathErrors',
               'skills', 'positive_skills', 'negative_skills','event_types', 'event_evaluations', 'stepTypes']
targets = ['finished']
ids = ['exerciseID', 'sessionID', 'questionID', 'interactionID']

descriptive_space = df[ids + descriptors]
target_space = df[ids + targets]

# all other attributes will be skipped
skipped = list( set(df.columns) - set(descriptors) - set(targets) - set(ids))

# get all data types of the descriptive attributes, necessary for refinement
descriptor_types = ca.manual_attribute_types()

# store relevant information on attribute placements
attributes = {'descriptive_attributes': descriptors, 'target_attributes': targets, 'id_attributes': ids, 'skip_attributes': skipped}

for attribute in cats[0].keys():
    condition = cats[0][attribute]
    print(condition)
    attr_type = [key for key, val in descriptor_types.items() if attribute in val][0]

#indices, subgroup_descriptors = ss.check_record_inclusion(descriptive_space=descriptive_space, dtypes=descriptor_types, candidate=cats[0])

('HINT', 'EVALUATE', 'EVALUATE')


In [6]:
selection = condition
print(selection)
selected_records = []

for i in range(len(descriptive_space)):
    attr_collection = descriptive_space[attribute].iloc[i]
    #print(f" == collection of row {i} = {attr_collection}")
    #print(f" == we look for {selection}")

    if selection[0] in attr_collection and len(selection) > 1:

        indices = [idx for idx, val in enumerate(attr_collection) if val == selection[0]]
        #print(f"= = we find {selection[0]} at {indices}")

        for id in indices:
            #print(f"= = we look into {id}")
                
            j = 1
            maximum = len(selection)  # we already checked the first one
            select = True

            if len(selection)  > ( len(attr_collection) - id ):
                #print(f"ERROR! TOO SHORT LENGTH OF {attr_collection} at {id} with {selection}")
                select = False
                continue

            while j < maximum: # go until all items are checked, or we find a mismatch
                    
                if attr_collection[id + j] == selection[j]:
                    #print(f"= = = = we find value {j}, i.e. {selection[j]} of {selection} ")
                    #print(f"= = = = this is found at {id + j} of {attr_collection}, i.e. {attr_collection[id + j]}")
                    j += 1
                    
                else:
                    #print(f"ERROR! NO MATCH")
                    select = False
                    j = maximum
                
            if select:
                #print(f"SUCCESS! MATCH FOUND")
                selected_records.append(i)
                continue #no need to go over the other indices

    elif selection[0] in attr_collection and len(selection) == 1:
        selected_records.append(i)
    
    else:
        continue

    #print(f"ERROR! NO MATCH FOR FIRST VARIABLE")
new_space = descriptive_space.iloc[selected_records]
new_space

('HINT', 'EVALUATE', 'EVALUATE')


,exerciseID,sessionID,questionID,interactionID,itemType,interactionType,marksPenalty,marksTotal,marksTotal_interaction,usedAttempts,marksPenalty,hintsRequested,mathErrors,skills,positive_skills,negative_skills,event_types,event_evaluations,stepTypes
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,Q1,Ex8wE,INTERACTION,MULTISTEP,0.35,4.0,1.0,1.0,0.35,1,1,[FractionArithmeticMultiplicationProper],[],[],"[EVALUATE, HINT, EVALUATE, EVALUATE]","[ERROR, HINT, PARSE_ERROR, FINISHED]",[CALCULATE]
124,9f4a675e-c2b3-4e11-91cc-4445951bab60,3bafc1c3-e40b-43f1-b7e0-201f1b14e187,Q1,Ex8wE,INTERACTION,MULTISTEP,0.00,4.0,1.0,2.0,0.00,1,2,[FractionArithmeticMultiplicationProper],[],[],"[HINT, EVALUATE, EVALUATE, GIVEUP]","[HINT, ERROR, ERROR, GIVEUP]",[CALCULATE]
161,9f4a675e-c2b3-4e11-91cc-4445951bab60,5376fbd8-5c8c-4827-9b98-3938b5e6e8cc,Q2,gBslt,INTERACTION,MULTISTEP,0.35,4.0,1.0,1.0,0.35,1,1,[FractionArithmeticMultiplicationProper],[],[],"[EVALUATE, HINT, EVALUATE, EVALUATE]","[ERROR, HINT, CORRECT, FINISHED]",[CALCULATE]
261,9f4a675e-c2b3-4e11-91cc-4445951bab60,83754b8f-a851-49b6-9a4d-9c35abade968,Q2,gBslt,INTERACTION,MULTISTEP,0.00,4.0,1.0,2.0,0.00,1,2,[FractionArithmeticMultiplicationProper],[],[],"[HINT, EVALUATE, EVALUATE, GIVEUP]","[HINT, ERROR, ERROR, GIVEUP]",[CALCULATE]
301,9f4a675e-c2b3-4e11-91cc-4445951bab60,9a4402d3-56ec-4cb0-b159-cbfca1729a1e,Q2,gBslt,INTERACTION,MULTISTEP,0.00,4.0,1.0,2.0,0.00,1,2,[FractionArithmeticMultiplicationProper],[],"[FractionArithmeticMultiplicationProper, Fract...","[HINT, EVALUATE, EVALUATE, GIVEUP]","[HINT, ERROR, ERROR, GIVEUP]",[CALCULATE]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6617,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f345b561-7e91-423a-81cf-6d9bb3eb120b,Q7,HuaWk,INTERACTION,MULTISTEP,0.00,6.0,1.0,0.0,0.00,1,0,"[EquationsBalanceMethodArrangeTerms, Equations...",[],[],"[EVALUATE, EVALUATE, EVALUATE, EVALUATE, EVALU...","[UNKNOWN_STEP, UNKNOWN_STEP, UNKNOWN_STEP, UNK...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]"
6623,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f40a0899-a745-4fc2-9727-7a3401c4d6c7,Q7,HuaWk,INTERACTION,MULTISTEP,0.00,6.0,1.0,2.0,0.00,1,2,"[EquationsBalanceMethodArrangeTerms, Equations...",[],[],"[EVALUATE, EVALUATE, EVALUATE, EVALUATE, EVALU...","[UNKNOWN_STEP, PARSE_ERROR, PARSE_ERROR, UNKNO...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]"
6624,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f40a0899-a745-4fc2-9727-7a3401c4d6c7,Q8,lwOZO,INTERACTION,MULTISTEP,0.00,6.0,1.0,2.0,0.00,1,2,"[EquationsBalanceMethodArrangeTerms, Equations...",[],[],"[EVALUATE, EVALUATE, HINT, EVALUATE, EVALUATE,...","[UNKNOWN_STEP, UNKNOWN_STEP, HINT, NO_MATCH, N...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]"
6626,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f5ad2de1-c604-4326-8c8a-d13ff9726879,Q4,Thm0S,INTERACTION,MULTISTEP,0.00,6.0,1.0,2.0,0.00,1,2,"[EquationsBalanceMethodArrangeTerms, Equations...",[],[],"[EVALUATE, HINT, EVALUATE, EVALUATE, GIVEUP]","[NO_MATCH, HINT, UNKNOWN_STEP, NO_MATCH, GIVEUP]","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]"
